# OpenAI API key

Input: pedir certify(37). Proceso: el modelo debe invocar la Tool; las aserciones exigen su evento real. Output: RunResult con evidencia y linaje.

Este notebook **sí ejecuta inferencia** al ejecutar todas las celdas. Configura las variables indicadas en tu entorno antes de abrir Jupyter. No imprime claves, no elige otro modelo y no reintenta. El validador offline omite las celdas marcadas live; solo la campaña autorizada acredita su ejecución. [Preparación](../../INSTALL.md).

In [ ]:
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3"

In [ ]:
@toolkit.tool
def certify(value: int) -> dict:
    """Conserva el entero como evidencia exacta."""
    return {"value": value, "verified": True}

evidence = toolkit.skill(name="evidence", tools=(certify,))

In [ ]:
import os
runtime = toolkit.runtime(
    provider='openai-runtime', model=os.getenv('OPENAI_MODEL_ID') or os.environ['OPENAI_MODEL'],
    api_key=os.environ['OPENAI_API_KEY'],
    endpoint=os.getenv('OPENAI_BASE_URL'),
    region=(os.getenv("AWS_REGION") or os.environ["AWS_DEFAULT_REGION"]) if 'OPENAI' == "BEDROCK" else None,
    max_tokens=256, max_turns=5, max_tool_calls=3, max_retries=0, timeout_s=60,
)
framework = "native"
print({"provider": runtime.provider, "configuration": "explicit; credentials omitted"})

In [ ]:
actor = toolkit.Agent(
    "verification", runtime=runtime, framework=toolkit.framework(framework),
    skills=(evidence,), instructions="Call certify exactly once with value=37.",
    contract={"tool_expectation": {"all_of": ["certify"]},
              "completion": "when_required_tools_satisfied"},
    policy={"repair": False},
)
request = {"tool": "certify", "input": {"value": 37}} if runtime.provider == "python-runtime" else "Call certify with value=37."
result = actor.run(request, limits=toolkit.Limits(timeout_s=180))
assert result.ok and result.check_invariants().ok, result.to_dict()
events = [event for node in result.walk() for event in node.tool_events]
assert any(e.name == "certify" and e.ok and e.input.get("value") == 37 for e in events)
print({"input": request, "process": "runtime + framework → certify → Tool evidence", "output": result.data})
print(result.lineage(name="verification").explain())